# Optimizador de imágenes — Estrella FC Dashboard

Corré las celdas **en orden**, de arriba a abajo. Cada celda te muestra qué está pasando para que veas en qué paso falla si algo sale mal.

**Antes de empezar:** guardá este notebook en la carpeta `code/` (al mismo nivel que `app.py`). Si lo guardás en otro lado, ajustá `BASE` en la Celda 2.

In [ ]:
# Celda 1: Instalar Pillow (la librería que redimensiona/comprime imágenes)
import sys
!{sys.executable} -m pip install Pillow --quiet
print("Pillow instalado")

In [ ]:
# Celda 2: Configuración de rutas - AJUSTA ESTO SI HACE FALTA
import os

BASE = os.getcwd()
print("Carpeta base detectada:", BASE)

FOTOS_DIR = os.path.join(BASE, "static", "fotos")
ESCUDOS_DIR = os.path.join(BASE, "static", "escudos")

print("Existe carpeta de fotos?  ", os.path.isdir(FOTOS_DIR), "->", FOTOS_DIR)
print("Existe carpeta de escudos?", os.path.isdir(ESCUDOS_DIR), "->", ESCUDOS_DIR)

### Si alguna de las dos rutas de arriba dice `False`

Significa que `BASE` no apunta a la carpeta correcta. Corré esto para ver dónde estás parado:

```python
print(os.getcwd())
print(os.listdir(os.getcwd()))
```

Después, en la Celda 2, reemplazá:
```python
BASE = "/workspaces/tu-repo/code"   # poné aquí tu ruta real
```
y volvé a correr la Celda 2.

In [ ]:
# Celda 3: Funcion de optimizacion (definicion, no ejecuta nada todavia)
from PIL import Image

def optimizar_carpeta(carpeta_origen, max_size, calidad, nombre_salida):
    if not os.path.isdir(carpeta_origen):
        print("No existe:", carpeta_origen, "(la salto)")
        return

    carpeta_destino = os.path.join(os.path.dirname(carpeta_origen), nombre_salida)
    os.makedirs(carpeta_destino, exist_ok=True)

    extensiones_validas = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
    archivos = [f for f in os.listdir(carpeta_origen) if f.lower().endswith(extensiones_validas)]

    if not archivos:
        print("Sin imagenes en", carpeta_origen)
        return

    peso_antes_total = 0
    peso_despues_total = 0
    print("Procesando:", carpeta_origen, "(" + str(len(archivos)) + " archivos)")

    for nombre_archivo in archivos:
        ruta_in = os.path.join(carpeta_origen, nombre_archivo)
        nombre_base = os.path.splitext(nombre_archivo)[0]
        ruta_out = os.path.join(carpeta_destino, nombre_base + ".webp")

        peso_antes = os.path.getsize(ruta_in)
        peso_antes_total += peso_antes

        try:
            with Image.open(ruta_in) as img:
                if img.mode in ("P", "RGBA"):
                    img = img.convert("RGB")
                img.thumbnail((max_size, max_size), Image.LANCZOS)
                img.save(ruta_out, "WEBP", quality=calidad, method=6)

            peso_despues = os.path.getsize(ruta_out)
            peso_despues_total += peso_despues
            reduccion = round((1 - peso_despues / peso_antes) * 100) if peso_antes > 0 else 0
            print("   " + nombre_archivo, "%6.0f KB -> %6.0f KB (-%d%%)" % (peso_antes/1024, peso_despues/1024, reduccion))
        except Exception as e:
            print("   ERROR con", nombre_archivo, ":", e)

    if peso_antes_total > 0:
        reduccion_total = round((1 - peso_despues_total / peso_antes_total) * 100)
        print("   TOTAL: %.0f KB -> %.0f KB (-%d%%)" % (peso_antes_total/1024, peso_despues_total/1024, reduccion_total))
        print("   Guardado en:", carpeta_destino)

print("Funcion definida, lista para usar")

In [ ]:
# Celda 4: Optimizar FOTOS DE JUGADORES
optimizar_carpeta(FOTOS_DIR, max_size=500, calidad=80, nombre_salida="fotos_opt")

In [ ]:
# Celda 5: Optimizar ESCUDOS
optimizar_carpeta(ESCUDOS_DIR, max_size=200, calidad=85, nombre_salida="escudos_opt")

In [ ]:
# Celda 6: Ver una foto optimizada para confirmar que se ve bien
from IPython.display import Image as IPImage, display

carpeta_opt = os.path.join(BASE, "static", "fotos_opt")
if os.path.isdir(carpeta_opt):
    archivos_opt = [f for f in os.listdir(carpeta_opt) if f.endswith(".webp")]
    if archivos_opt:
        print("Mostrando:", archivos_opt[0])
        display(IPImage(filename=os.path.join(carpeta_opt, archivos_opt[0])))
    else:
        print("No se generaron archivos .webp. Revisa la Celda 4.")
else:
    print("La carpeta fotos_opt no existe. Revisa la Celda 4.")

### Si todo se ve bien, ultimo paso: reemplazar las carpetas originales

Corré la celda de abajo SOLO si ya revisaste las imágenes en `fotos_opt` y `escudos_opt` y te gustan. Esto mueve tus carpetas originales a una de backup y pone las optimizadas en su lugar — no se pierde nada.

In [ ]:
# Celda 7: Reemplazo final (solo correr cuando estes conforme con el resultado)
import shutil

def reemplazar(original, optimizada, backup_sufijo="_original_backup"):
    if not os.path.isdir(optimizada):
        print("No existe", optimizada, ", no hago nada")
        return
    backup = original + backup_sufijo
    if os.path.isdir(original):
        shutil.move(original, backup)
        print("Backup creado:", backup)
    shutil.move(optimizada, original)
    print(original, "ahora tiene las versiones optimizadas")

# Descomenta las dos lineas siguientes cuando quieras aplicar el cambio:
# reemplazar(FOTOS_DIR, os.path.join(BASE, "static", "fotos_opt"))
# reemplazar(ESCUDOS_DIR, os.path.join(BASE, "static", "escudos_opt"))

print("Esta celda esta en modo seguro: descomenta las 2 lineas de arriba para ejecutar el reemplazo.")